# Heston calibration and structured-product workflow

This notebook contains market inputs and orchestration only. Reusable logic is
in `heston_calibration.py` and `heston_pricing.py`; the volatility workbooks are
kept beside this notebook. Run it with the **spyder-env** kernel and use the
notebook directory as the working directory.

Historical regularization is configured independently for each index below.
Replace `historical_params` with the immediately preceding trading day's
calibration. Set `use_historical=False` to run the original calibration.


In [1]:
from pathlib import Path

import numpy as np

from heston_calibration import (
    DividendCarryCurve,
    calibrate_heston_gatheral,
    historical_parameter_report,
    load_vol_surface,
    run_no_arbitrage_checks,
)
from heston_pricing import (
    calculate_greeks,
    price_autocallable_heston,
    price_phoenix_heston,
    price_snowball_heston,
)


## Market and calibration configuration

`history_penalty_iv=0.002` makes an average one-tolerance parameter move cost
the same as 0.2 volatility points of IV RMSE. The default tolerances allow more
movement in `v0` than in the structural parameters. The historical point is
also supplied to differential evolution when `warm_start_from_history=True`.


In [2]:
DATA_DIR = Path.cwd()
required_files = ["vol_surface1000.xlsx", "vol_surface300.xlsx"]
missing = [name for name in required_files if not (DATA_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(
        f"Missing {missing}. Set DATA_DIR to the equity folder containing this notebook."
    )

# Update historical_params after each successful daily calibration.
INDEX_CONFIG = {
    "CSI 1000": {
        "surface": "vol_surface1000.xlsx",
        "spot": 7587.75,
        "rate": 0.014,
        "effective_yields": np.array(
            [0.044, 0.1299, 0.116, 0.1256, 0.1237, 0.119]
        ),
        "fix_kappa": True,
        "kappa_fixed": 6.0,
        "stage1_fixed_rho": -0.2,
        "stage2_penalty": 1.0,
        "weight_by_T": True,
        "use_historical": False,
        "historical_params": {'v0': 0.10489785362073611, 'kappa': 7.107013627475046, 'theta': 0.077061501689074, 'sigma': 1.5142755676094402, 'rho': -0.22591245046438188},
        "history_penalty_iv": 0.002,
        "history_tolerances": None,
        "warm_start_from_history": True,
        "use_historical_rho_in_stage1": True,
    },
    "CSI 300": {
        "surface": "vol_surface300.xlsx",
        "spot": 4548.39,
        "rate": 0.014,
        "effective_yields": np.array(
            [0.0549, 0.0889, 0.0727, 0.0817, 0.079, 0.0722]
        ),
        "fix_kappa": True,
        "kappa_fixed": 6.0,
        "stage1_fixed_rho": -0.2,
        "stage2_penalty": 1.0,
        "weight_by_T": True,
        "use_historical": False,
        "historical_params": {'v0': 0.032984085860661094, 'kappa': 6.0, 'theta': 0.04752294666383868, 'sigma': 1.4263621026552802, 'rho': -0.16964525706442785},
        "history_penalty_iv": 0.002,
        "history_tolerances": None,
        "warm_start_from_history": False,
        "use_historical_rho_in_stage1": False,
    },
}


In [3]:
def calibrate_index(config):
    market = load_vol_surface(DATA_DIR / config["surface"])
    maturities = np.sort(market["T"].unique())
    curve = DividendCarryCurve(maturities, config["effective_yields"])
    params, fitted = calibrate_heston_gatheral(
        market,
        config["rate"],
        curve,
        config["spot"],
        fix_kappa=config["fix_kappa"],
        kappa_fixed=config["kappa_fixed"],
        stage1_fixed_rho=config["stage1_fixed_rho"],
        stage2_penalty=config["stage2_penalty"],
        weight_by_T=config["weight_by_T"],
        use_historical=config["use_historical"],
        historical_params=config["historical_params"],
        history_penalty_iv=config["history_penalty_iv"],
        history_tolerances=config["history_tolerances"],
        warm_start_from_history=config["warm_start_from_history"],
        use_historical_rho_in_stage1=config["use_historical_rho_in_stage1"],
    )
    return market, curve, params, fitted


def show_calibration_summary(config, params, fitted):
    print("Calibrated parameters:")
    for name, value in params.items():
        print(f"  {name}: {value:.10f}")

    if config["use_historical"]:
        active_names = ["v0", "kappa", "theta", "sigma", "rho"]
        if config["fix_kappa"]:
            active_names.remove("kappa")
        print("\nMovement from historical parameters:")
        print(
            historical_parameter_report(
                params,
                config["historical_params"],
                history_tolerances=config["history_tolerances"],
                active_names=active_names,
            ).to_string(index=False)
        )

    print("\nCalibration diagnostics:")
    print(fitted.attrs["calibration_diagnostics"])


## CSI 1000 calibration

In [4]:
config_1000 = INDEX_CONFIG["CSI 1000"]
market_1000, q_curve_1000, params_1000, fitted_1000 = calibrate_index(
    config_1000
)

print(q_curve_1000.knot_table(S0=config_1000["spot"], r=config_1000["rate"]))
show_calibration_summary(config_1000, params_1000, fitted_1000)
fitted_1000.head(10)


d:\LuJingjian\Jupyter_files\GuoYuan\equity\HST pricer\heston_calibration.py:374: NumbaPerformanceWarning: '@' is faster on contiguous arrays, called on (Array(float64, 1, 'C', False, aligned=True), Array(float64, 2, 'A', False, aligned=True))
  int1 = wts @ integrand1


Stage 1 complete: v0=0.1006, kappa=6.0000, theta=0.0763, sigma=1.8792, fixed rho=-0.2000
Stage 2 complete: v0=0.1058, kappa=6.0000, theta=0.0644, sigma=1.8798, rho=-0.1785
          T  q_effective  Q_cumulative  dividend_discount      forward
0  0.028689       0.0440      0.001262           0.998739  7581.222369
1  0.086066       0.1299      0.011180           0.988882  7512.438435
2  0.188525       0.1160      0.021869           0.978369  7443.235226
3  0.270492       0.1256      0.033974           0.966597  7362.122097
4  0.512295       0.1237      0.063371           0.938595  7173.088638
5  0.770492       0.1190      0.091689           0.912389  6998.063483
Calibrated parameters:
  v0: 0.1058329222
  kappa: 6.0000020000
  theta: 0.0644044614
  sigma: 1.8798103087
  rho: -0.1784839116

Calibration diagnostics:
{'unpenalized_iv_mse': 0.011657975010253955, 'unpenalized_iv_rmse': 0.10797210292595932, 'history_enabled': False, 'history_penalty_iv': 0.0, 'history_distance': 0.0, 'history_

,T,K,IV,q_used,Q_cumulative,Forward,Fitted_IV,IV_error
0,0.028689,6200,0.4497,0.044,0.001262,7581.222369,0.411811,-0.037889
1,0.028689,6300,0.4382,0.044,0.001262,7581.222369,0.403683,-0.034517
2,0.028689,6400,0.4266,0.044,0.001262,7581.222369,0.395520,-0.031080
3,0.028689,6500,0.4146,0.044,0.001262,7581.222369,0.387324,-0.027276
4,0.028689,6600,0.4025,0.044,0.001262,7581.222369,0.379102,-0.023398
5,0.028689,6700,0.3900,0.044,0.001262,7581.222369,0.370866,-0.019134
6,0.028689,6800,0.3773,0.044,0.001262,7581.222369,0.362636,-0.014664
7,0.028689,6900,0.3641,0.044,0.001262,7581.222369,0.354442,-0.009658
8,0.028689,7000,0.3506,0.044,0.001262,7581.222369,0.346334,-0.004266
9,0.028689,7100,0.3366,0.044,0.001262,7581.222369,0.338387,0.001787


In [5]:
run_no_arbitrage_checks(
    market_1000,
    params_1000,
    config_1000["spot"],
    config_1000["rate"],
    q_curve_1000,
    "000852 CSI 1000",
)


000852 CSI 1000   (S0=7587.75, r=0.014, maturity-specific q)

Dividend curve used at quoted maturities
    T=0.028689: q_eff=4.400000%  Q=0.00126230  F=7581.22
    T=0.086066: q_eff=12.990000%  Q=0.01117992  F=7512.44
    T=0.188525: q_eff=11.600000%  Q=0.02186885  F=7443.24
    T=0.270492: q_eff=12.560000%  Q=0.03397377  F=7362.12
    T=0.512295: q_eff=12.370000%  Q=0.06337090  F=7173.09
    T=0.770492: q_eff=11.900000%  Q=0.09168852  F=6998.06

[0] true martingale check rho <= kappa/sigma
    rho - kappa/sigma = -3.3703

[1] Forward via characteristic function vs S0 exp(rT-Q(T))
    T=0.029:  CF=   7581.22  theory=   7581.22  rel=1.59e-12
    T=0.086:  CF=   7512.44  theory=   7512.44  rel=5.45e-12
    T=0.189:  CF=   7443.24  theory=   7443.24  rel=1.68e-11
    T=0.270:  CF=   7362.12  theory=   7362.12  rel=3.37e-11
    T=0.512:  CF=   7173.09  theory=   7173.09  rel=1.95e-10
    T=0.770:  CF=   6998.06  theory=   6998.06  rel=1.07e-09
    -> max rel fwd error = 1.07e-09   PASS

[

## CSI 300 calibration

In [ ]:
config_300 = INDEX_CONFIG["CSI 300"]
market_300, q_curve_300, params_300, fitted_300 = calibrate_index(config_300)

print(q_curve_300.knot_table(S0=config_300["spot"], r=config_300["rate"]))
show_calibration_summary(config_300, params_300, fitted_300)
fitted_300.head(10)


In [ ]:
run_no_arbitrage_checks(
    market_300,
    params_300,
    config_300["spot"],
    config_300["rate"],
    q_curve_300,
    "000300 CSI 300",
)

## Snowball pricing and Greeks example

In [6]:
spot = config_1000["spot"]
obs_times = np.arange(21, 21 * 12 + 1, 21)  # trading days
pay_times = np.arange(30, 30 * 12 + 1, 30)  # 30/360 calendar days

snowball_inputs = {
    "S0": spot,
    "T": 1.0,
    "coupon": .2854,
    "knock_out": spot,
    "H_in": 0.9 * spot,
    "obs_times": obs_times,
    "r": config_1000["rate"],
    "q": q_curve_1000,
    "kappa": params_1000["kappa"],
    "theta": params_1000["theta"],
    "sigma_v": params_1000["sigma"],
    "rho": params_1000["rho"],
    "v0": params_1000["v0"],
    "Ns": 1000,
    "Nv": 40,
    "Nt_uniform": 252,
    "daily_ki": True,
    "SPD": 0.8,
    "pay_times": pay_times,
}
M = 1000000
snowball_price = price_snowball_heston(**snowball_inputs)
snowball_greeks = calculate_greeks(price_snowball_heston, snowball_inputs)
print(f"Price per unit notional: {snowball_price * M:.8f}")
print(f"delta: {snowball_greeks['delta'] * M:.8f}")
snowball_greeks

Price per unit notional: -19850.49515251
delta: 61.70377491


{'delta': 6.170377491059189e-05,
 'gamma': 5.157207740968868e-07,
 'vega': -0.1297348510530797}

In [ ]:
spot = config_300["spot"]
obs_times = np.arange(21, 21 * 24 + 1, 21)  # trading days
pay_times = np.arange(30, 30 * 24 + 1, 30)  # 30/360 calendar days

snowball_inputs = {
    "S0": spot,
    "T": 2.0,
    "coupon": 0.068,
    "knock_out": spot * np.arange(1.01,1.01-24*0.005,-0.005),
    "H_in": 0.8 * spot,
    "obs_times": obs_times,
    "r": config_300["rate"],
    "q": 0.06,#q_curve_300,
    "kappa": params_300["kappa"],
    "theta": params_300["theta"],
    "sigma_v": params_300["sigma"],
    "rho": params_300["rho"],
    "v0": 0.03,#params_300["v0"],
    "Ns": 1000,
    "Nv": 40,
    "Nt_uniform": 504,
    "daily_ki": True,
    "SPD": 0.85,
    "pay_times": pay_times,
}
M = 4000000
snowball_price = price_snowball_heston(**snowball_inputs)
snowball_greeks = calculate_greeks(price_snowball_heston, snowball_inputs)
print(f"Price per unit notional: {snowball_price * M:.8f}")
print(f"delta: {snowball_greeks['delta'] * M:.8f}")
snowball_greeks

In [ ]:
spot = config_1000["spot"]
obs_times = np.arange(21, 21 * 24 + 1, 21)  # trading days
pay_times = np.arange(30, 30 * 24 + 1, 30)  # 30/360 calendar days

autocallable_inputs = {
    "S0": spot,
    "T": 2.0,
    "coupon": 0.0278,
    "knock_out": spot * 1.01,
    "obs_times": obs_times,
    "premium": 0.01,
    "r": config_1000["rate"],
    "q": 0.08,#q_curve_1000,
    "kappa": params_1000["kappa"],
    "theta": params_1000["theta"],
    "sigma_v": params_1000["sigma"],
    "rho": params_1000["rho"],
    "v0": params_1000["v0"],
    "Ns": 1000,
    "Nv": 40,
    "Nt_uniform": 504,
    "pay_times": pay_times,
    "mg": 0,
    "mg_ist": None,
    "q_maturities": None,
}
M = 10000000
auto_price = price_autocallable_heston(**autocallable_inputs)
snowball_greeks = calculate_greeks(price_autocallable_heston, autocallable_inputs)
print(f"Price per unit notional: {auto_price * M:.8f}")
print(f"delta: {snowball_greeks['delta'] * M:.8f}")
snowball_greeks

In [ ]:
spot = config_300["spot"]
obs_times = np.arange(21, 21 * 12 + 1, 21)  # trading days
pay_times = np.arange(30, 30 * 12 + 1, 30)  # 30/360 calendar days

autocallable_inputs = {
    "S0": spot,
    "T": 1.0,
    "coupon": 0.0265,
    "knock_out": spot * 1.01,
    "obs_times": obs_times,
    "premium": 0.01,
    "r": config_300["rate"],
    "q": 0.06,#q_curve_300,
    "kappa": params_300["kappa"],
    "theta": params_300["theta"],
    "sigma_v": params_300["sigma"],
    "rho": params_300["rho"],
    "v0": params_300["v0"],
    "Ns": 1000,
    "Nv": 40,
    "Nt_uniform": 252,
    "pay_times": pay_times,
    "mg": 0,
    "mg_ist": None,
    "q_maturities": None,
}
M = 10000000
auto_price = price_autocallable_heston(**autocallable_inputs)
snowball_greeks = calculate_greeks(price_autocallable_heston, autocallable_inputs)
print(f"Price per unit notional: {auto_price * M:.8f}")
print(f"delta: {snowball_greeks['delta'] * M:.8f}")
snowball_greeks

In [ ]:
spot = config_1000["spot"]
obs_times = np.arange(21, 21 * 24 + 1, 21)/252  # trading days

phoenix_inputs = {
    "S0":spot,
    "T": 2.,
    "coupon": 0.0067,
    "knock_out": spot,
    "H_in": 0.8*spot,
    "obs_times": obs_times,
   "r": config_1000["rate"],
    "q": 0.07,
    "kappa": params_1000["kappa"],
    "theta": params_1000["theta"],
    "sigma_v": params_1000["sigma"],
    "rho": params_1000["rho"],
    "v0": 0.06,#params_1000["v0"],
    "Ns": 1000,
    "Nv": 40,
    "Nt_uniform": 504,
    "H_coupon": None,
    "daily_ki": False,
    "SPD": 0.8,
    "mg": 0,
    "mg_ist": None,
    "full_term":None,
    "K":None,
    "q_maturities":None,
}
M = 30000000
phoe_price = price_phoenix_heston(**phoenix_inputs)
# snowball_greeks = calculate_greeks(price_snowball_heston, snowball_inputs)
print(f"Price per unit notional: {phoe_price * M:.8f}")
# snowball_greeks

In [ ]:
spot = config_300["spot"]
obs_times = np.arange(21, 21 * 6 + 1, 21)/252  # trading days

phoenix_inputs = {
    "S0":spot,
    "T": .5,
    "coupon": 0.0068,
    "knock_out": spot,
    "H_in": 0.85*spot,
    "obs_times": obs_times,
   "r": config_300["rate"],
    "q": 0.06,#q_curve_300,
    "kappa": params_300["kappa"],
    "theta": params_300["theta"],
    "sigma_v": params_300["sigma"],
    "rho": params_300["rho"],
    "v0": params_300["v0"],
    "Ns": 1000,
    "Nv": 40,
    "Nt_uniform": 126,
    "H_coupon": None,
    "daily_ki": True,
    "SPD": 0.85,
    "mg": 0,
    "mg_ist": None,
    "full_term":None,
    "K":None,
    "q_maturities":None,
}
M = 3000000
phoe_price = price_phoenix_heston(**phoenix_inputs)
# snowball_greeks = calculate_greeks(price_snowball_heston, snowball_inputs)
print(f"Price per unit notional: {phoe_price * M:.8f}")
# snowball_greeks

In [ ]:
np.sqrt(params_300["v0"])

In [ ]:
0.24*0.25